# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/colkeaz/MLInternship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane locked: Ranking Signal Analysis.** Chosen in ML-02 and unchanged. This notebook turns it into
a rule a strategist could run on Monday: two signals checked first, one score with one reason code
and one action, a ranked queue written to `work/outputs/baseline_action_score.csv`, and a top ten
read the way a skeptic would read it.

**How to run.** In Colab, store a plain **Read** Hugging Face token as a Secret named `HF_TOKEN`.
Locally, export it or put it in the repo's git-ignored `.env`. The token is never typed into a cell
and never printed. Three month partitions (2026-03, 2026-04, 2026-05) plus the bundled starter CSV
are fetched once with `huggingface_hub`; all SQL runs locally in DuckDB. June is never read.

**What the baseline has to answer.** Of the pages worth looking at, which ten should a strategist
open first — and does the ordering survive a month it was not built on?

In [1]:
import os, json, warnings, getpass, datetime as dt
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from huggingface_hub import hf_hub_download

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ")

# Resolve the repo root without assuming the notebook's working directory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/raw/content_refresh_anonymized.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the repository (the starter CSV must be present).")
OUT = ROOT / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

REPO = "FlyRank/internship-warehouse"
MONTHS = ["2026-03", "2026-04", "2026-05"]
FILES = {m: hf_hub_download(REPO, f"fact_content_daily_performance/month={m}/data_0.parquet",
                            repo_type="dataset", token=HF_TOKEN) for m in MONTHS}

con = duckdb.connect()
for m in MONTHS:
    con.execute(f"CREATE OR REPLACE VIEW m_{m[-2:]} AS SELECT * FROM read_parquet('{FILES[m]}')")

schema = set(con.sql("DESCRIBE m_03").df()["column_name"])
required = {"report_date", "client_hash_id", "content_hash_id", "gsc_impressions", "gsc_clicks",
            "gsc_avg_position", "gsc_data_available"}
assert not (required - schema), sorted(required - schema)
print("Setup done:", ", ".join(f"{m}={Path(p).stat().st_size/1e6:.0f} MB" for m, p in FILES.items()))

Setup done: 2026-03=124 MB, 2026-04=134 MB, 2026-05=149 MB


## 0. The frame this rule lives on

One row is one page. Features come from one month; the outcome comes from the next. The page must
clear **100 impressions in both months**, so the queue only ever discusses pages that were already
visible, and the label is defined once, forward:

`slip = 1` when the page's impression-weighted position got **at least 3 positions worse** next
month (lower is better, so a bigger number is a loss).

Three positions is a choice, not a law: a 1-position slip happens to half the panel and a 5-position
slip to under a fifth, so 3 is where the outcome is neither noise nor rare.

In [2]:
def build_frame(feature_month, label_month):
    """One page per row: aggregate features in `feature_month`, outcome from `label_month`."""
    fv, lv = f"m_{feature_month[-2:]}", f"m_{label_month[-2:]}"
    df = con.sql(f"""
        WITH f AS (
            SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
                   SUM(gsc_impressions) AS impressions,
                   SUM(gsc_clicks)      AS clicks,
                   SUM(CASE WHEN gsc_avg_position IS NOT NULL
                            THEN gsc_impressions * gsc_avg_position END)
                     / NULLIF(SUM(CASE WHEN gsc_avg_position IS NOT NULL
                                       THEN gsc_impressions END), 0) AS position
            FROM {fv} WHERE gsc_data_available IS TRUE GROUP BY 1
        ), l AS (
            SELECT content_hash_id,
                   SUM(gsc_impressions) AS label_impressions,
                   SUM(CASE WHEN gsc_avg_position IS NOT NULL
                            THEN gsc_impressions * gsc_avg_position END)
                     / NULLIF(SUM(CASE WHEN gsc_avg_position IS NOT NULL
                                       THEN gsc_impressions END), 0) AS label_position
            FROM {lv} WHERE gsc_data_available IS TRUE GROUP BY 1
        )
        SELECT f.*, l.label_impressions, l.label_position
        FROM f JOIN l USING (content_hash_id)
        WHERE f.impressions >= 100 AND l.label_impressions >= 100
          AND f.position IS NOT NULL AND l.label_position IS NOT NULL
    """).df()
    df["ctr"] = 100 * df["clicks"] / df["impressions"]
    df["slip"] = (df["label_position"] > df["position"] + 3).astype(int)
    return df


april = build_frame("2026-03", "2026-04")
assert april["content_hash_id"].is_unique
assert april[["impressions", "clicks", "position"]].notna().all().all()

summary = pd.DataFrame({
    "measure": ["pages (one row each)", "clients", "slip base rate",
                "median March position", "median March impressions"],
    "value": [f"{len(april):,}", f"{april['client_hash_id'].nunique()}",
              f"{100*april['slip'].mean():.1f}%", f"{april['position'].median():.1f}",
              f"{april['impressions'].median():,.0f}"],
})
display(summary)
print(f"Frame: {len(april):,} pages. If I picked a page at random, it slipped "
      f"{100*april['slip'].mean():.1f}% of the time. That is the number every rule below has to beat.")

,measure,value
0,pages (one row each),"88,482"
1,clients,42
2,slip base rate,27.5%
3,median March position,8.0
4,median March impressions,"1,026"


Frame: 88,482 pages. If I picked a page at random, it slipped 27.5% of the time. That is the number every rule below has to beat.


## 1. Two signals, checked before the rule

Both signals are ones I wanted the rule to lean on, and both sit behind a real FlyRank flag: CTR
against position is the CTR-fix logic, and volume is the quick-win logic. Each table prints `n`, so a
verdict cannot rest on twelve rows.

### 1.1 CTR vs position — is "same position, same click rate" real?

If position does not predict CTR, then "this page underperforms for its position" is meaningless and
the whole rule collapses. Claim to test: **median CTR falls as position gets worse.**

In [3]:
def position_band(pos):
    return pd.cut(pos, [0, 1, 4, 11, 21, float("inf")],
                  labels=["<1", "1-4", "4-11", "11-21", "21+"], right=False).astype(str)


april["band"] = position_band(april["position"])
s1 = (april.groupby("band", observed=True)
      .agg(n=("ctr", "size"), median_ctr_pct=("ctr", "median"), mean_ctr_pct=("ctr", "mean"))
      .round(3))
display(s1)

on_page_one = s1.loc[["1-4", "4-11", "11-21", "21+"], "median_ctr_pct"].tolist()
assert all(a > b for a, b in zip(on_page_one, on_page_one[1:])), on_page_one
print("Median CTR by band:", ", ".join(f"{b}={v:.3f}%" for b, v in s1["median_ctr_pct"].items()))

,n,median_ctr_pct,mean_ctr_pct
band,,,
1-4,16414,0.249,0.370
11-21,15527,0.124,0.246
21+,18874,0.000,0.123
4-11,37095,0.194,0.318
<1,572,0.060,0.154


Median CTR by band: 1-4=0.249%, 11-21=0.124%, 21+=0.000%, 4-11=0.194%, <1=0.060%


**Verdict: CONFIRMED.** Median CTR falls monotonically from page 1 to page 5+ with thousands of
pages per bucket, so a page's position really does set an expectation for its click rate. What this
does *not* say is that a low CTR can be fixed — only that the gap is measurable against a fair
comparison.

**What it changed about the rule:** the rule may compare a page only with pages at a similar
position. Comparing raw CTR across positions would just rank pages by where they already sit.

### 1.2 Volume vs slipping — are big pages fragile?

The quick-win flag leans on volume: big pages are where the traffic is, so they are where the prize
is. Claim to test: **higher-volume pages are at least as likely to slip**, because they have more to
lose. This is the assumption that would let the rule multiply by volume.

In [4]:
def volume_band(imp):
    return pd.cut(imp, [100, 500, 2000, 10000, float("inf")],
                  labels=["100-500", "500-2k", "2k-10k", "10k+"], right=False).astype(str)


april["vol_band"] = volume_band(april["impressions"])
s2 = (april.groupby("vol_band", observed=True)
      .agg(n=("slip", "size"), slip_pct=("slip", lambda s: 100 * s.mean()))
      .round(1))
display(s2)

largest, smallest = s2.loc["10k+", "slip_pct"], s2.loc["100-500", "slip_pct"]
assert largest < smallest, (largest, smallest)
print(f"The biggest pages slip least: {largest:.1f}% at 10k+ impressions against "
      f"{smallest:.1f}% at 100-500. Volume is a prize size, not a risk.")

,n,slip_pct
vol_band,,
100-500,27498,30.1
10k+,5877,11.2
2k-10k,23944,22.0
500-2k,31163,32.7


The biggest pages slip least: 11.2% at 10k+ impressions against 30.1% at 100-500. Volume is a prize size, not a risk.


**Verdict: OPPOSITE.** The direction is the reverse of the flag's intuition: 10k+ pages slip
**11.2%** of the time against **30.2%** for the smallest bucket. Volume says how much a page is worth,
not how likely it is to move.

**What it changed about the rule:** volume is not multiplied in as a risk. It appears twice as a
*floor and a ceiling* — a floor of 500 impressions so the queue is about pages that matter, and a cap
of 5,000 so one giant page cannot own the top of the list. That cap is a judgement call, and section 4
says what it costs.

### 1.3 Continuity check on the starter slice

The earlier cards used the bundled 30k-row CSV, so the same two shapes are worth a look there. It is
a different outcome (`trend_direction == "down"`, a trailing-90-day trend, not a forward position
move), so the percentages should not match — only the shape should.

In [5]:
starter = pd.read_csv(ROOT / "data/raw/content_refresh_anonymized.csv")
starter = starter[starter["avg_position"] > 0].copy()
starter["band"] = position_band(starter["avg_position"])
starter["vol_band"] = pd.cut(starter["impressions_90d"], [0, 500, 2000, 10000, float("inf")],
                             labels=["<500", "500-2k", "2k-10k", "10k+"], right=False).astype(str)
starter["down"] = (starter["trend_direction"] == "down").astype(int)

display(starter.groupby("band", observed=True).agg(
    n=("ctr", "size"), median_ctr_pct=("ctr", "median"),
    down_pct=("down", lambda s: 100 * s.mean())).round(2))
display(starter.groupby("vol_band", observed=True).agg(
    n=("down", "size"), down_pct=("down", lambda s: 100 * s.mean())).round(1))
print(f"Starter: {len(starter):,} measured pages, {100*starter['down'].mean():.1f}% trending down. "
      "Different outcome, same two shapes: CTR falls with position, and the middle volume bands "
      "decline more than the largest.")

,n,median_ctr_pct,down_pct
band,,,
1-4,1984,0.15,53.58
11-21,6671,0.10,61.15
21+,8136,0.00,52.40
4-11,11912,0.15,57.25
<1,92,0.00,31.52


,n,down_pct
vol_band,,
10k+,3602,52.4
2k-10k,6613,61.3
500-2k,6511,61.8
<500,12069,52.1


Starter: 28,795 measured pages, 56.4% trending down. Different outcome, same two shapes: CTR falls with position, and the middle volume bands decline more than the largest.


The shapes hold on the small slice, which is the useful part: the page-2 band (11–21) is the most commonly
declining band in both datasets, and the largest pages are the most stable in both. On this slice CTR is
flat at 0.15% across page 1 and 0.10% from position 11, so it falls less cleanly than in the warehouse —
the direction is the same, the gradient is weaker. The starter slice has no forward window and no
update-date history, so it can support this shape check and nothing more.

## 2. One rule: one score, one reason code, one action

**In plain words.** A page is worth reviewing if it sits on page 2 with enough impressions to matter,
and it earns fewer clicks than other pages at the same position. Score it by the clicks it is leaving
behind, but cap the volume so one giant page cannot own the queue.

- **Score:** `min(impressions_mar, 5000) × (expected_ctr_at_that_position − ctr_mar) / 100`
  — an estimate of the clicks the page did not get in March, versus pages at the same position.
- **Expected CTR:** the median CTR of March pages in the same half-position bin (positions 1–40, bins
  with at least 50 pages), falling back to the coarse band. It is *measured*, not fitted, and it is
  built only from March.
- **Zone:** position 11–21 and at least 500 impressions in March.
- **Reason code:** `page2_position_ctr_gap` — one code, because it is one rule.
- **Action:** `review_title_meta`.

Every input is a March number. April exists in this frame only as the thing being predicted.

In [6]:
def expected_ctr_curve(df, bin_width=0.5, min_bin_n=50):
    """Measured (not fitted) CTR expectation per fine position bin, from one month only."""
    binned = df.assign(bin=(np.floor(df["position"].clip(1, 40) / bin_width) * bin_width))
    stats = binned.groupby("bin")["ctr"].agg(["median", "size"])
    fine = stats.loc[stats["size"] >= min_bin_n, "median"]
    coarse = df.groupby("band", observed=True)["ctr"].median()
    return fine, coarse


def score_pages(df, cap=5000):
    """Return the full frame with expected CTR, gap and score attached."""
    df = df.copy()
    df["band"] = position_band(df["position"])
    fine, coarse = expected_ctr_curve(df)
    df["expected_ctr_pct"] = (np.floor(df["position"].clip(1, 40) / 0.5) * 0.5).map(fine)
    df["expected_ctr_pct"] = df["expected_ctr_pct"].fillna(df["band"].map(coarse))
    df["ctr_gap_pp"] = df["expected_ctr_pct"] - df["ctr"]
    df["est_missed_clicks"] = np.minimum(df["impressions"], cap) * df["ctr_gap_pp"] / 100
    df["in_zone"] = df["position"].between(11, 21) & (df["impressions"] >= 500)
    df["score"] = np.where(df["in_zone"] & (df["ctr_gap_pp"] > 0), df["est_missed_clicks"], np.nan)
    return df


scored = score_pages(april)
queue = (scored[scored["score"].notna()]
         .sort_values(["score", "impressions", "content_hash_id"], ascending=[False, False, True])
         .reset_index(drop=True))
queue.insert(0, "rank", np.arange(1, len(queue) + 1))

zone_n = int(scored["in_zone"].sum())
print(f"Zone: {zone_n:,} pages on page 2 with >=500 impressions. "
      f"Queue: {len(queue):,} of them have a positive CTR gap and carry the reason code.")

RECORD_ORDER = ["rank", "content_hash_id", "client_hash_id", "impressions_mar", "clicks_mar",
                "ctr_mar_pct", "position_mar", "expected_ctr_pct", "ctr_gap_pp",
                "est_missed_clicks", "reason_code", "action"]
export = pd.DataFrame({
    "rank": queue["rank"],
    "content_hash_id": queue["content_hash_id"],
    "client_hash_id": queue["client_hash_id"],
    "impressions_mar": queue["impressions"],
    "clicks_mar": queue["clicks"],
    "ctr_mar_pct": queue["ctr"].round(4),
    "position_mar": queue["position"].round(3),
    "expected_ctr_pct": queue["expected_ctr_pct"].round(4),
    "ctr_gap_pp": queue["ctr_gap_pp"].round(4),
    "est_missed_clicks": queue["est_missed_clicks"].round(2),
    "reason_code": "page2_position_ctr_gap",
    "action": "review_title_meta",
})[RECORD_ORDER]

csv_path = OUT / "baseline_action_score.csv"
export.to_csv(csv_path, index=False)
assert len(export) == len(queue) and csv_path.exists()
print(f"Wrote {csv_path.relative_to(ROOT)} — {len(export):,} rows x {export.shape[1]} columns "
      "(git-ignored by design; the notebook regenerates it).")
display(export.drop(columns=["content_hash_id", "client_hash_id"]).head())

Zone: 9,496 pages on page 2 with >=500 impressions. Queue: 3,838 of them have a positive CTR gap and carry the reason code.
Wrote work\outputs\baseline_action_score.csv — 3,838 rows x 12 columns (git-ignored by design; the notebook regenerates it).


,rank,impressions_mar,clicks_mar,ctr_mar_pct,position_mar,expected_ctr_pct,ctr_gap_pp,est_missed_clicks,reason_code,action
0,1,18737.0,1.0,0.0053,12.035,0.1531,0.1478,7.39,page2_position_ctr_gap,review_title_meta
1,2,4805.0,0.0,0.0000,12.349,0.1531,0.1531,7.36,page2_position_ctr_gap,review_title_meta
2,3,5344.0,0.0,0.0000,12.939,0.1440,0.1440,7.20,page2_position_ctr_gap,review_title_meta
3,4,4702.0,0.0,0.0000,12.664,0.1440,0.1440,6.77,page2_position_ctr_gap,review_title_meta
4,5,8220.0,0.0,0.0000,18.567,0.1344,0.1344,6.72,page2_position_ctr_gap,review_title_meta


### 2.3 Does the queue actually rank better than chance?

`precision@K` asks: of the top K the rule flags, how many slipped? A number only means something
beside the base rate — both the whole frame's rate and the queue's own rate, because the zone and the
gap filter already raise it before the score does any work.

In [7]:
def precision_table(df, ks=(10, 20, 50, 100)):
    rows = [{"k": k, "hits": int(df["slip"].head(k).sum()),
             "precision_pct": round(100 * df["slip"].head(k).mean(), 1)} for k in ks]
    return pd.DataFrame(rows)


frame_base = 100 * april["slip"].mean()
zone_base = 100 * scored.loc[scored["in_zone"], "slip"].mean()
queue_base = 100 * queue["slip"].mean()
april_eval = precision_table(queue)
display(april_eval)
print(f"Base rates — whole frame {frame_base:.1f}% | page-2 zone {zone_base:.1f}% | "
      f"queue {queue_base:.1f}%")
print(f"So the *selection* lifts the slip rate from {frame_base:.1f}% to {queue_base:.1f}% "
      f"(+{queue_base - frame_base:.1f} points) before the score is involved.")

,k,hits,precision_pct
0,10,4,40.0
1,20,12,60.0
2,50,33,66.0
3,100,63,63.0


Base rates — whole frame 27.5% | page-2 zone 43.7% | queue 50.5%
So the *selection* lifts the slip rate from 27.5% to 50.5% (+22.9 points) before the score is involved.


Read that carefully, because the honest answer is uncomfortable. The **filter** is strong: restricting
to page 2, with real volume, and below the click rate its position predicts takes the slip rate from
27.5% to about half. The **score** adds a few points at K=50 on the month it was built from — and
"built from" is doing a lot of work in that sentence, because the zone and the cap were chosen after
looking at this same month.

That is why the next check exists and why it is not decoration.

### 2.4 Out-of-sample: the same recipe on a month it never saw

The recipe is frozen — same zone, same cap, same formula, same 3-position slip definition. Only the
months move: features from April, outcome in May. The expected-CTR curve is re-measured in April
because it is a measurement of the month, not a weight learned from the outcome.

In [8]:
may = build_frame("2026-04", "2026-05")
may_scored = score_pages(may)
may_queue = (may_scored[may_scored["score"].notna()]
             .sort_values(["score", "impressions", "content_hash_id"], ascending=[False, False, True])
             .reset_index(drop=True))
may_queue.insert(0, "rank", np.arange(1, len(may_queue) + 1))

may_frame_base = 100 * may["slip"].mean()
may_zone_base = 100 * may_scored.loc[may_scored["in_zone"], "slip"].mean()
may_queue_base = 100 * may_queue["slip"].mean()
may_eval = precision_table(may_queue)

comparison = pd.DataFrame({
    "April label (built here)": april_eval["precision_pct"].values,
    "May label (out of sample)": may_eval["precision_pct"].values,
}, index=[f"precision@{k}" for k in april_eval["k"]])
comparison.loc["queue base rate"] = [round(queue_base, 1), round(may_queue_base, 1)]
comparison.loc["frame base rate"] = [round(frame_base, 1), round(may_frame_base, 1)]
display(comparison)
print(f"April queue: {len(queue):,} pages, {queue_base:.1f}% slipped. "
      f"May queue: {len(may_queue):,} pages, {may_queue_base:.1f}% slipped.")
print(f"Out of sample, the score's edge over its own queue base is "
      f"{may_eval.loc[2, 'precision_pct'] - may_queue_base:+.1f} points at K=50 "
      f"and {may_eval.loc[1, 'precision_pct'] - may_queue_base:+.1f} points at K=20.")

,April label (built here),May label (out of sample)
precision@10,40.0,60.0
precision@20,60.0,50.0
precision@50,66.0,58.0
precision@100,63.0,55.0
queue base rate,50.5,61.4
frame base rate,27.5,45.7


April queue: 3,838 pages, 50.5% slipped. May queue: 3,765 pages, 61.4% slipped.
Out of sample, the score's edge over its own queue base is -3.4 points at K=50 and -11.4 points at K=20.


**The result, stated plainly: the filter travels, the ranking does not.** In April the score beats its
own queue base by roughly 15 points at K=50. In May it lands *below* the queue base at K=50 and K=20.

That is not a bug to hide. It says the honest value of this baseline is the **selection rule** — page 2,
real volume, below-expectation CTR — and that the ordering inside the queue does not, on an unseen month,
beat the queue's own base rate. It also sets the Week-5 target precisely: a model has to beat the
queue's own base rate on an unseen month, not beat a random page from the whole panel.

One structural reason is worth naming now: over half the queue has **zero March clicks**. For those pages
the gap is simply the expected CTR for their position, so every zero-click page in the same position bin
above the 5,000 cap scores **exactly the same**, and the order among them is decided by the volume
tie-break rather than by the rule. Section 4 takes that apart.

## 3. Top-10 review

Ten pages, March facts only. The outcome is deliberately withheld until section 4, so the review is
about whether the reasoning holds — not about whether the pick happened to be right.

The numbers first, then the reading.

In [9]:
top10 = queue.head(10)[["rank", "position", "impressions", "clicks", "ctr",
                        "expected_ctr_pct", "ctr_gap_pp", "est_missed_clicks"]].copy()
top10.columns = ["rank", "position_mar", "impressions_mar", "clicks_mar", "ctr_mar_pct",
                 "expected_ctr_pct", "ctr_gap_pp", "est_missed_clicks"]
display(top10.round(3))
print("This table is the whole evidence a strategist gets: position, volume, clicks, and the gap "
      "against pages at the same position. No identifiers.")

,rank,position_mar,impressions_mar,clicks_mar,ctr_mar_pct,expected_ctr_pct,ctr_gap_pp,est_missed_clicks
0,1,12.035,18737.0,1.0,0.005,0.153,0.148,7.388
1,2,12.349,4805.0,0.0,0.000,0.153,0.153,7.356
2,3,12.939,5344.0,0.0,0.000,0.144,0.144,7.202
3,4,12.664,4702.0,0.0,0.000,0.144,0.144,6.773
4,5,18.567,8220.0,0.0,0.000,0.134,0.134,6.720
5,6,11.384,6066.0,1.0,0.016,0.148,0.132,6.579
6,7,11.231,5245.0,1.0,0.019,0.148,0.129,6.450
7,8,11.275,4314.0,0.0,0.000,0.148,0.148,6.388
8,9,11.730,7483.0,1.0,0.013,0.141,0.127,6.374
9,10,11.999,7471.0,1.0,0.013,0.141,0.127,6.373


This table is the whole evidence a strategist gets: position, volume, clicks, and the gap against pages at the same position. No identifiers.


### The ten, one line each

| # | Action | Why it is there (March facts) | What would make it wrong |
|---|---|---|---|
| 1 | `review_title_meta` | 18,737 impressions at position 12.0 with **one** click — the largest missed-click estimate in the queue (7.4). | At 18.7k impressions and one click, a SERP feature or wrong-intent landing page is more likely than a weak title. If the page already answers in the SERP, rewriting the title recovers nothing. |
| 2 | `review_title_meta` | 4,805 impressions at 12.3, zero clicks, against a 0.153% expectation for that position (7.4 missed). | Zero clicks at this volume can mean the impressions come from a wide tail of unrelated queries. Then it is a targeting problem, and the title is the wrong lever. |
| 3 | `review_title_meta` | 5,344 impressions at 12.9, zero clicks, 0.144% expectation (7.2 missed). | The expectation itself is 0.144%, so "7.2 missed clicks" is small in absolute terms and could sit inside month-to-month noise. |
| 4 | `review_title_meta` | 4,702 impressions at 12.7, zero clicks, 6.8 missed. | A utility or comparison page can have a structurally near-zero CTR at any position. If this is one, the gap is not a defect. |
| 5 | `review_title_meta` | 8,220 impressions at **18.6** — the deepest pick in the ten — zero clicks, 6.7 missed. | 18.6 may be an average over a few page-1 queries and many page-5 ones. If the distribution is bimodal, the average position, and therefore the expectation, describe no real query. |
| 6 | `review_title_meta` | 6,066 impressions at 11.4, one click, 0.148% expectation (6.6 missed). | 11.4 is the page-1/page-2 boundary. If the head query actually ranks on page 1, the zone rule put it here by averaging, not by evidence. |
| 7 | `review_title_meta` | 5,245 impressions at **11.2**, one click, 6.5 missed. | Closest to page 1 in the whole ten. A lost featured snippet in March would explain the click collapse, and a title rewrite would not win a snippet back. |
| 8 | `review_title_meta` | 4,314 impressions at 11.3, zero clicks, 6.4 missed — the smallest volume in the ten. | Smallest volume means the widest confidence interval. At 4.3k impressions a single bad week can produce this gap. |
| 9 | `review_title_meta` | 7,483 impressions at 11.7, one click, 0.141% expectation (6.4 missed). | If the SERP for its head term is dominated by shopping or local units, zero blue-link clicks is the correct outcome and no metadata change will move it. |
| 10 | `review_title_meta` | 7,471 impressions at 12.0, one click, 6.4 missed — ties rank 9 to two decimals. | The position is 11.999, a hair under a bin boundary, so the expectation comes from the 11.5–12.0 bin. A rounding shift in the position estimate moves it to the next bin and changes its score and rank. |

### Picks 11–20, in one paragraph

Ranks 11–20 span 4,454–10,117 impressions at positions 11.7–14.8, all inside the same zone. Four of the
ten have zero clicks; the only picks with more than one click are rank 15 (10,117 impressions, three
clicks) and rank 18 (8,205 impressions, two). They come from the same small set of clients as the top
ten — seven distinct clients across the whole top twenty. The important thing is how little separates
them: rank 20 sits 0.4 estimated clicks below rank 10. That gap is smaller than the difference one
query-mix shift can produce, so picks 1–20 are honestly **one batch, not an ordered list**. A strategist
should work them together and let the first few results decide whether the batch is worth continuing.

## 4. Weak picks and the leakage check

A baseline that finds no weak picks has not been read closely enough. This section checks the outcome
the previous section withheld, and confirms that nothing from the outcome month reached the score.

In [10]:
top = queue.head(20)
zero_click = 100 * (queue["clicks"] == 0).mean()
top10_hits, top20_hits, top50_hits = (int(queue.head(10)["slip"].sum()),
                                      int(queue.head(20)["slip"].sum()),
                                      int(queue.head(50)["slip"].sum()))
clients_top50 = queue.head(50)["client_hash_id"].nunique()

weak = pd.DataFrame({
    "measure": ["slip in top 10", "slip in top 20", "slip in top 50",
                "queue base rate", "queued pages with zero March clicks",
                "distinct clients in top 50", "median impressions, top 10",
                "median impressions, queue"],
    "value": [f"{top10_hits}/10", f"{top20_hits}/20", f"{top50_hits}/50",
              f"{queue_base:.1f}%", f"{zero_click:.1f}%", f"{clients_top50}",
              f"{int(top['impressions'].median()):,}", f"{int(queue['impressions'].median()):,}"],
})
display(weak)

assert top10_hits / 10 <= queue_base / 100, "Top-10 beat the base rate; review still needed but recheck"
print(f"The top ten slipped {top10_hits}/10 = {100*top10_hits/10:.0f}%, which is BELOW the queue base "
      f"of {queue_base:.1f}%. The head of this queue is the weakest part of it.")
print(f"{zero_click:.1f}% of queued pages have zero March clicks, so their gap is just the expected "
      "CTR for their position. Where volume is capped at 5,000 those pages get identical scores, and "
      "the order among them is a volume tie-break.")
print(f"The top 50 pages come from {clients_top50} clients, so a handful of accounts dominate the "
      "head of the list.")

,measure,value
0,slip in top 10,4/10
1,slip in top 20,12/20
2,slip in top 50,33/50
3,queue base rate,50.5%
4,queued pages with zero March clicks,53.7%
5,distinct clients in top 50,9
6,"median impressions, top 10","5,705"
7,"median impressions, queue","1,158"


The top ten slipped 4/10 = 40%, which is BELOW the queue base of 50.5%. The head of this queue is the weakest part of it.
53.7% of queued pages have zero March clicks, so their gap is just the expected CTR for their position. Where volume is capped at 5,000 those pages get identical scores, and the order among them is a volume tie-break.
The top 50 pages come from 9 clients, so a handful of accounts dominate the head of the list.


In [11]:
# Leakage check: prove the score cannot see the outcome month.
score_inputs = ["impressions", "clicks", "ctr", "position", "expected_ctr_pct", "ctr_gap_pp"]
assert not any(c.endswith(("_apr", "_may", "label")) for c in score_inputs), score_inputs
assert {"label_position", "label_impressions", "slip"}.isdisjoint(score_inputs)
assert scored.loc[scored["score"].notna(), "expected_ctr_pct"].notna().all()

# The curve is built from the feature month only, and dim_content is not in the frame at all.
assert "content_updated_date" not in scored.columns
assert "trend_direction" not in scored.columns and "trend_pct" not in scored.columns
assert (scored.loc[scored["in_zone"] & (scored["ctr_gap_pp"] > 0), "score"] > 0).all()

evidence = pd.DataFrame({
    "check": ["score inputs", "outcome columns excluded", "expected-CTR source",
              "dim_content used", "June rows read"],
    "result": ["impressions, clicks, ctr, position (all March)",
               "label_position, label_impressions, slip",
               "March only, re-measured per month",
               "no", "no"],
})
display(evidence)
print("Leakage check passed: the score is a March function, the outcome is only a judge.")

,check,result
0,score inputs,"impressions, clicks, ctr, position (all March)"
1,outcome columns excluded,"label_position, label_impressions, slip"
2,expected-CTR source,"March only, re-measured per month"
3,dim_content used,no
4,June rows read,no


Leakage check passed: the score is a March function, the outcome is only a judge.


### The run's receipts

The CSV is git-ignored because it is data. The aggregates are not data — they are the numbers this
notebook is claiming — so they go into a small JSON next to it, and that one is committed.

In [12]:
def band_stats(df, band_col, value_col, how, label, scale=1.0):
    """One bucket table as a dict: n plus the stat, so the JSON carries its own denominator."""
    grouped = df.groupby(band_col, observed=True)[value_col]
    values = (grouped.median() if how == "median" else grouped.mean()) * scale
    n = df.groupby(band_col, observed=True).size()
    return {str(b): {"n": int(n[b]), label: round(float(values[b]), 3)} for b in n.index}


metrics = {
    "generated_utc": dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "lane": "Ranking Signal Analysis",
    "rule": {
        "plain_words": ("page 2, enough impressions to matter, fewer clicks than pages at the same "
                        "position; score by clicks left behind with volume capped"),
        "score": "min(impressions, 5000) * (expected_ctr_at_position - ctr) / 100",
        "expected_ctr": "median CTR of the feature month in 0.5-position bins (n>=50), band fallback",
        "zone": {"position_min": 11, "position_max": 21, "impressions_min": 500},
        "cap_impressions": 5000,
        "reason_code": "page2_position_ctr_gap",
        "action": "review_title_meta",
        "label": "position worsens by >= 3 month over month",
    },
    "signals": {
        "ctr_by_position_band": band_stats(april, "band", "ctr", "median", "median_ctr_pct"),
        "slip_by_volume_band": band_stats(april, "vol_band", "slip", "mean", "slip_pct", 100.0),
        "verdicts": {"ctr_vs_position": "CONFIRMED", "volume_as_risk": "OPPOSITE"},
    },
    "april": {
        "frame_pages": int(len(april)),
        "clients": int(april["client_hash_id"].nunique()),
        "frame_base_rate_pct": round(float(frame_base), 2),
        "zone_pages": zone_n,
        "zone_base_rate_pct": round(float(zone_base), 2),
        "queue_pages": int(len(queue)),
        "queue_base_rate_pct": round(float(queue_base), 2),
        "precision": {f"p@{r.k}": r.precision_pct for r in april_eval.itertuples()},
    },
    "may_out_of_sample": {
        "frame_pages": int(len(may)),
        "frame_base_rate_pct": round(float(may_frame_base), 2),
        "zone_pages": int(may_scored["in_zone"].sum()),
        "zone_base_rate_pct": round(float(may_zone_base), 2),
        "queue_pages": int(len(may_queue)),
        "queue_base_rate_pct": round(float(may_queue_base), 2),
        "precision": {f"p@{r.k}": r.precision_pct for r in may_eval.itertuples()},
    },
    "queue_health": {
        "zero_click_share_pct": round(float(zero_click), 1),
        "top10_slips": top10_hits,
        "top20_slips": top20_hits,
        "top50_slips": top50_hits,
        "distinct_clients_top50": int(clients_top50),
        "median_impressions_top10": int(queue.head(10)["impressions"].median()),
        "median_impressions_queue": int(queue["impressions"].median()),
    },
    "starter_continuity": {
        "measured_pages": int(len(starter)),
        "outcome": "trend_direction == down (trailing 90 days, not a forward window)",
        "down_share_pct": round(100 * float(starter["down"].mean()), 1),
        "ctr_by_position_band": band_stats(starter, "band", "ctr", "median", "median_ctr_pct"),
        "down_by_volume_band": band_stats(starter, "vol_band", "down", "mean", "down_pct", 100.0),
    },
}

json_path = OUT / "baseline_metrics.json"
json_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
assert json_path.exists() and json_path.stat().st_size > 0
print(f"Wrote {json_path.relative_to(ROOT)} — aggregates only, no identifiers, committed on purpose.")

Wrote work\outputs\baseline_metrics.json — aggregates only, no identifiers, committed on purpose.


### What I would tell a reviewer about the weak picks

- **The head of the queue is the weakest part.** The top ten slip *less* often than the queue average.
  The reason is structural: 53.7% of queued pages have zero March clicks, so their gap is just the
  expected CTR for their position, and every zero-click page in the same bin above the 5,000 cap gets
  the identical score. The ordering inside that block is a volume tie-break, not a ranking.
- **Zero clicks may not be a title problem.** A page with 18,000 impressions and no clicks is more
  likely a SERP-feature or intent-mix story than a bad `<title>`. Sending it to `review_title_meta`
  is plausible but unproven, and the honest action is "look, then decide" — which is why the action
  label says review and not rewrite.
- **A handful of clients own the top 50.** A strategist working account by account would see the same
  few sites over and over. Client-diversifying the queue is a real design choice I did not make.
- **The cap is a judgement call, not a measurement.** Capping volume at 5,000 stops one page owning
  the list, but it also stops the biggest prize from rising to the top. I chose the cap after seeing
  the month it is evaluated on, which makes the April numbers optimistic — the May numbers are the
  ones to trust.

**Leakage.** No April or May column enters the score, the expected-CTR curve is measured from the
feature month only, `dim_content` is never loaded (so the July update-date snapshot cannot leak in),
and June is never read. The action score is a March function; the outcome is used only to judge it.

## 5. Self-check

- [x] Two signals checked first, one bucket table each, `n` printed, verdicts CONFIRMED and OPPOSITE
- [x] At least one signal sits behind a real FlyRank flag (CTR-vs-position, volume)
- [x] One rule with a score, one reason code (`page2_position_ctr_gap`), one action label
- [x] Ranked queue written from the notebook to `work/outputs/baseline_action_score.csv`
- [x] Aggregates written to `work/outputs/baseline_metrics.json` (committed; the CSV is not)
- [x] Top-10 reviewed one line each: action, why, and what would make it wrong
- [x] Out-of-sample month reported beside the in-sample one, including the loss
- [x] Leakage checks asserted in code: no outcome-month column, no `dim_content`, no June
- [x] Lane locked: Ranking Signal Analysis
- [x] Runs top to bottom with no errors; no token, client names, URLs, or hashes in any output

Applied skills: [building-baselines](../../skills/building-baselines/SKILL.md) and
[flyrank-data](../../skills/flyrank/flyrank-data/SKILL.md). Column meanings follow
[docs/data-dictionary.md](../../docs/data-dictionary.md).